In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 데이터 로드
X_full = pd.read_csv('data/train.csv', index_col='Id')
X_test_full = pd.read_csv('data/test.csv', index_col='Id')

X_full.info()

# 타켓이 없는 행 제거, 타겟 분리
X_full.dropna(subset=['SalePrice'], axis=0, inplace=True)
y = X_full.SalePrice
X_full.drop(['SalePrice'], axis=1, inplace=True)

# 숫자 데이터만 사용
X = X_full.select_dtypes(exclude=['object'])
X_test = X_test_full.select_dtypes(exclude=['object'])

# 훈련 및 검증 데이터 분리
X_train, X_valid, y_train, y_valid = train_test_split (
    X, y, train_size=0.8, test_size=0.2, random_state=0
)

<class 'pandas.core.frame.DataFrame'>
Index: 1460 entries, 1 to 1460
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MSSubClass     1460 non-null   int64  
 1   MSZoning       1460 non-null   object 
 2   LotFrontage    1201 non-null   float64
 3   LotArea        1460 non-null   int64  
 4   Street         1460 non-null   object 
 5   Alley          91 non-null     object 
 6   LotShape       1460 non-null   object 
 7   LandContour    1460 non-null   object 
 8   Utilities      1460 non-null   object 
 9   LotConfig      1460 non-null   object 
 10  LandSlope      1460 non-null   object 
 11  Neighborhood   1460 non-null   object 
 12  Condition1     1460 non-null   object 
 13  Condition2     1460 non-null   object 
 14  BldgType       1460 non-null   object 
 15  HouseStyle     1460 non-null   object 
 16  OverallQual    1460 non-null   int64  
 17  OverallCond    1460 non-null   int64  
 18  YearBuilt    

### 데이터 확인
- 결측값 제거

In [42]:
X_train.isnull().sum()

# 결측 데이터 있는 열 찾기
missing = [col for col in X_train.columns 
           if X_train[col].isnull().any()
        ]

# 결측 데이터 삭제
reduced_X_train = X_train.drop(missing, axis=1)
reduced_X_valid = X_valid.drop(missing, axis=1)

### 모델 성능 평가

[방법1] : 누락된 열 삭제

In [43]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 모델 성능 평가 함수
def score_dataset(X_train, X_valid, y_train, y_valid) :
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)

    return mean_absolute_error(y_valid, preds)

print("MAE (누락된 열 삭제):")
print(score_dataset(reduced_X_train, reduced_X_valid, y_train, y_valid))

MAE (누락된 열 삭제):
17837.82570776256


[방법2] : 평균값으로 대체

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')
imputed_X_train = pd.DataFrame(imputer.fit_transform(X_train))
imputed_X_valid = pd.DataFrame(imputer.transform(X_valid))

# 원래 열 이름 복원
imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns

print("MAE (평균값 대체):")
print(score_dataset(imputed_X_train, imputed_X_valid, y_train, y_valid))

MAE (평균값 대체):
18062.894611872147


[방법3] : 중앙값으로 대체

In [51]:
# 중앙값으로 대체
final_imputer = SimpleImputer(strategy='median')
final_X_train = pd.DataFrame(final_imputer.fit_transform(X_train))
final_X_valid = pd.DataFrame(final_imputer.transform(X_valid))

# 열 이름 복원
final_X_train.columns = X_train.columns
final_X_valid.columns = X_valid.columns

# 모델 학습 및 예측
model = RandomForestRegressor(n_estimators=100, random_state=0)
model.fit(final_X_train, y_train)
preds_valid = model.predict(final_X_valid)

print("MAE (중앙값 대체):")
print(mean_absolute_error(y_valid, preds_valid))


MAE (중앙값 대체):
17791.59899543379


### 결과
중앙값으로 대체 > 누락된 열 삭제 > 평균값으로 대체 순으로 성능이 좋음을 확인할 수 있습니다. 